        # 01 希望条件とマッチング

        **F3：出会いとマッチングのアルゴリズム**（Stage 1：Matching Formation の準備）

        - 架空の参加者の**希望条件**から**希望スコア**と**希望順位**を作る
        - どの組が**相互承諾可能**（双方が相手を受け入れる）かを確かめる
        - 計算：`participants.py`・`preferences.py`

> **教育用の架空データ・架空モデル**です。参加者は実在の人物ではなく、Group A・B はアルゴリズム上の区分で性別や性的指向を意味しません。希望スコアは実際の結婚適性を表さず、関係継続モデルの係数は実際の離婚率や実証された因果効果ではありません。

## 準備：実験環境（最初に必ず実行）

F3 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F3 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F3 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F3 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                              # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f3-matching-algorithms"     # リポジトリの中の F3 フォルダ（ルートが F3 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f3_root(p):
    return (Path(p) / "matching_algorithms.py").exists() and (Path(p) / "relationship_stability.py").exists()


def find_f3_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F3 のルート
        if is_f3_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f3_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F3 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f3_upload")
    for hit in glob.glob("/content/f3_upload/**/matching_algorithms.py", recursive=True):
        if is_f3_root(Path(hit).parent):
            return Path(hit).parent
    return None


F3_ROOT = find_f3_root()
if F3_ROOT is None:
    raise RuntimeError("F3 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F3_ROOT)
if str(F3_ROOT) not in sys.path:
    sys.path.insert(0, str(F3_ROOT))
if not (F3_ROOT / "textbook" / "fixed_queue.py").exists():
    print("⚠️ textbook/ がありません。preferences.py・matching_algorithms.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import participants as pt
import preferences as pr
import matching_algorithms as ma
import relationship_stability as rs
import evaluation as ev
import matching_charts as mc

mc.setup_style()                       # 文字を大きくするなら mc.setup_style(1.3)
groups = pt.load_participants()
scores = pr.score_table(groups)
rankings = pr.build_rankings(groups, scores)
params = rs.load_parameters()          # 関係継続モデルのパラメータ（すべて架空）
A_IDS = [p["id"] for p in groups["A"]]
B_IDS = [p["id"] for p in groups["B"]]
N_TOTAL = pt.total_participants(groups)
compat = lambda a, b: pr.pair_compatibility(a, b, scores)

SAVE_FIGURES = False                   # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", mc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        heads = [h for _k, h, _f in columns]
        widths = [max(len(h), *(len(d[h]) for d in data)) for h in heads]
        print("  ".join(h.rjust(w) for h, w in zip(heads, widths)))
        for d in data:
            print("  ".join(d[h].rjust(w) for h, w in zip(heads, widths)))


print("F3_ROOT:", F3_ROOT)
print(f"参加者 Group A {len(A_IDS)} 人・Group B {len(B_IDS)} 人 / 関係継続モデル（架空）:",
      {k: params[k] for k in ("baseline_hazard", "alpha", "beta", "gamma", "default_annual_stress",
                               "default_conflict_resolution", "simulation_years", "random_seed")})

## 1. 架空の参加者

In [ ]:
table([dict(p, interests=", ".join(p["interests"]),
            lifestyle=", ".join(f"{k}={v}" for k, v in p["lifestyle"].items()),
            values=", ".join(f"{k}={v}" for k, v in p["values"].items()),
            age_range="{}–{}".format(*p["preferences"]["age_range"]),
            min_score=p["preferences"]["min_score"]) for g in ("A", "B") for p in groups[g]],
      [("id", "id", "{}"), ("age", "age", "{}"), ("interests", "interests", "{}"), ("lifestyle", "lifestyle", "{}"),
       ("values", "values", "{}"), ("age_range", "wants age", "{}"), ("min_score", "min score", "{}")])

## 2. 希望スコア（Compatibility Score）

$$ \text{Score}(p \to q) = w_1 \cdot \text{Interest} + w_2 \cdot \text{Lifestyle} + w_3 \cdot \text{Value} + w_4 \cdot \text{Age} $$

| 項目 | 計算（0〜1） |
| --- | --- |
| Interest | 趣味タグの Jaccard 係数（共通 ÷ 和集合） |
| Lifestyle・Value | 共通の項目について 1 − \|差\| の平均 |
| Age | 希望年齢の範囲内なら 1、範囲から d 歳離れると 1 − d/10 |

- 重み $w_1$〜$w_4$ は**その人自身の希望条件**（合計 1）。$p \to q$ と $q \to p$ でスコアは違う
- **希望スコアは教育用の人工的な指標**であり、実際の結婚適性を表すものではありません

In [ ]:
p, q = groups["A"][0], groups["B"][0]
print(f"{p['id']} から見た {q['id']}:", {k: round(v, 3) for k, v in pr.components(p, q).items()},
      "→ score", round(pr.compatibility_score(p, q), 3), " 重み", p["preferences"]["weights"])
show(mc.plot_score_heatmap(A_IDS, B_IDS, {(a, b): scores[(a, b)] for a in A_IDS for b in B_IDS}, rankings,
                           "Score from A's point of view (A -> B)"), "01_scores_a_to_b")

In [ ]:
#@title グラフを表示（コードは隠しています）
show(mc.plot_score_heatmap(A_IDS, B_IDS, {(a, b): scores[(b, a)] for a in A_IDS for b in B_IDS}, rankings,
                           "Score from B's point of view (B -> A)"), "01_scores_b_to_a")

## 3. 希望順位と相互承諾

- 受け入れる相手 = `acceptable_candidates` にいて、かつ **スコア ≥ min_score**
- 希望順位 = 受け入れる相手をスコアの高い順に（同点は ID 順）。教科書の**単純挿入ソート**（安定）
- **相互承諾可能** = 双方が相手を受け入れる組だけが、どのアルゴリズムでも成立し得る

In [ ]:
table([{"id": pid, "ranking": " > ".join(lst) or "(nobody)", "n": len(lst)} for pid, lst in rankings.items()],
      [("id", "participant", "{}"), ("ranking", "preference ranking (acceptable only)", "{}"), ("n", "#", "{}")])
pairs = ma.acceptable_pairs(rankings, A_IDS, B_IDS)
print(f"相互承諾可能な組：{len(pairs)} / {len(A_IDS) * len(B_IDS)}")

## 4. ペアの初期適合度

**初期適合度** = 2 方向の希望スコアの平均。成立したペアの関係継続モデル（Notebook 04）で使います。

In [ ]:
#@title グラフを表示（コードは隠しています）
show(mc.plot_score_heatmap(A_IDS, B_IDS, {(a, b): compat(a, b) for a in A_IDS for b in B_IDS}, rankings,
                           "Pair compatibility = mean of the two directions", "pair compatibility"),
     "01_pair_compatibility")

## 5. 学生向け TODO：希望条件の重みを変える

全員の重みを一時的に置き換えて、希望順位と相互承諾可能な組の数がどう変わるか調べます（元のデータは変更しません）。

In [ ]:
# TODO：重みを変えて実行しよう（合計 1）
new_weights = {"interest": 0.1, "lifestyle": 0.2, "values": 0.6, "age": 0.1}
s2 = pr.score_table(groups, weights_override={"A": new_weights, "B": new_weights})
r2 = pr.build_rankings(groups, s2)
for pid in ["A1", "A2", "B2"]:
    print(f"{pid}: 元の順位 {rankings[pid]}  →  新しい順位 {r2[pid]}")
print("相互承諾可能な組：", len(ma.acceptable_pairs(rankings, A_IDS, B_IDS)), "→", len(ma.acceptable_pairs(r2, A_IDS, B_IDS)))
# TODO：min_score を変える実験（RQ2）は experiments.py の Experiment A を参照

## 考察（チームで記入）

- **TODO**：重み・閾値（min_score）を変えると、誰が誰を受け入れるかはどう変わるか
- **TODO**：A から見たスコアと B から見たスコアが違うのはなぜか。それはマッチングにどう影響しそうか
- **TODO**：このスコアで「表せないもの」は何か（モデルの限界）

## 参考文献

- D. Gale, L. S. Shapley, "College Admissions and the Stability of Marriage", *The American Mathematical Monthly*, 69(1), 9–15, 1962.
- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第3章 線形探索・ハッシュ法、第4章 キュー、第6章 挿入ソート）
- F3 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_participants.json`、`data/relationship_parameters.json`（この教材のための**架空のデータ**）